In [116]:
import pandas as pd
import numpy as np
import os, glob
from tqdm import tqdm
import networkx as nx
from collections import defaultdict

from src.OrganoidMesh import OrganoidMesh
from src.mesh_analysis import *
from src.nonlocal_correlations import *
from src.cell_graph_functions import *
from src.utils import *

In [117]:
def load_cell_graph_from_npz(data: np.lib.npyio.NpzFile) -> nx.Graph:
    """
    Reconstruct a NetworkX graph from edges and node count stored in an NPZ file.
    This matches the new preprocessing format where `edges` and `n_nodes`
    are explicitly saved.
    """
    if "graph_edges" not in data.files or "graph_n_nodes" not in data.files:
        raise KeyError("NPZ file must contain 'edges' and 'n_nodes' to load the cell graph.")

    edges = data["graph_edges"]
    n_nodes = int(data["graph_n_nodes"])

    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))

    if edges.size > 0:
        edges = edges.reshape(-1, 2)
        G.add_edges_from(edges.tolist())

    return G


In [118]:
data_dir = '../NicoleData/20250929/fractal_output'

timepoint = "day4p5"
zarr_name = "r0.zarr"
well = "A06"
round_name = "0_fused_zillum_registered"
organoid_id = 31 # 11 #20 # 31 #19 # example

path = f"{data_dir}/{timepoint}/{zarr_name}/{well[0]}/{well[1:]}/{round_name}/meshes/nnorg_linked_multi_annotated_class/{organoid_id}.vtp"


mesh = OrganoidMesh()
mesh.load_mesh_from_file(path)
mesh.align_with_pca()
_ = mesh.compute_spectral_coefficients(lmax=15)    # must populate m.eigvals, m.eigvecs

print(mesh.v.shape)

cell_graph = build_cell_graph(mesh)

[Info] Eigen-decomposition not found. Computing now...
(18001, 3)


In [119]:
HKS_times = [1.0, 2.0, 4.0, 8, 12, 16, 20, 25.0]
hks, hks_coeffs = compute_hks(mesh, t=HKS_times,) # HKS at time-scale comparable to cell size


In [120]:
cell_center_vertices = mesh.get_centroid_vertices() # get indices of patch centers
centroids, cell_areas, cell_weighted_hks, _ = mesh.compute_cell_statistics(hks) # coarse-grain


In [121]:
dist_heat = compute_geodesics(mesh, t=None, sources=cell_center_vertices)
dist_heat = dist_heat[:, cell_center_vertices]

heat sources: 100%|██████████| 819/819 [00:03<00:00, 220.08it/s]


In [122]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

def plot_hks_graph_subplots(
    centroids, graph, fields_cell, scale_labels=None,
    colorscale="RdBu_r", max_cols=3, node_size=4, edge_width=1
):
    """
    Grid of 3D subplots (max 3 columns). Each subplot = HKS at one time scale
    with its own colorbar positioned within that subplot.
    """
    N, S = fields_cell.shape
    if scale_labels is None:
        scale_labels = [f"t[{i}]" for i in range(S)]

    ncols = min(max_cols, S)
    nrows = math.ceil(S / max_cols)

    # Precompute edges once
    edge_x, edge_y, edge_z = [], [], []
    for i, j in graph.edges():
        edge_x += [centroids[i, 0], centroids[j, 0], None]
        edge_y += [centroids[i, 1], centroids[j, 1], None]
        edge_z += [centroids[i, 2], centroids[j, 2], None]

    fig = make_subplots(
        rows=nrows, cols=ncols,
        specs=[[{'type': 'scene'} for _ in range(ncols)] for _ in range(nrows)],
        subplot_titles=scale_labels
    )

    for k in range(S):
        row = k // ncols + 1
        col = k % ncols + 1

        field = fields_cell[:, k]
        vmax = float(np.nanmax(np.abs(field))) if np.any(np.isfinite(field)) else 1.0
        cmin, cmax = -vmax, vmax  # center at 0 per subplot

        # ---- colorbar position for THIS subplot (normalized figure coords) ----
        # Subplot domain: [ (col-1)/ncols , col/ncols ] × [ 1-row/nrows , 1-(row-1)/nrows ]
        x0 = (col - 1) / ncols
        x1 = col / ncols
        y0 = 1 - row / nrows
        y1 = 1 - (row - 1) / nrows
        cb_x = x1 - 0.02                 # a bit inside the right edge of this subplot
        cb_y = (y0 + y1) / 2.0           # vertically centered in this subplot
        cb_len = 0.65 / nrows            # short bar; scales with number of rows

        # Edges
        fig.add_trace(
            go.Scatter3d(
                x=edge_x, y=edge_y, z=edge_z,
                mode="lines",
                line=dict(width=edge_width, color="lightgray"),
                hoverinfo="none",
                showlegend=False
            ),
            row=row, col=col
        )

        # Nodes
        fig.add_trace(
            go.Scatter3d(
                x=centroids[:, 0], y=centroids[:, 1], z=centroids[:, 2],
                mode="markers",
                marker=dict(
                    size=node_size,
                    color=field,
                    colorscale=colorscale,
                    cmin=cmin, cmax=cmax,
                    colorbar=dict(
                        title="",
                        x=cb_x, y=cb_y,
                        yanchor="middle",
                        len=cb_len,
                        thickness=12
                    ),
                    line=dict(width=0)
                ),
                hoverinfo="none",
                showlegend=False,
                name=scale_labels[k]
            ),
            row=row, col=col
        )

        # Keep aspect ratio
        scene_name = "scene" if (row == 1 and col == 1) else f"scene{k+1}"
        fig.update_layout(**{scene_name: dict(aspectmode='data')})

    fig.update_layout(
        height=500 * nrows,
        width=380 * ncols,
        margin=dict(l=10, r=10, t=40, b=10),
        showlegend=False
    )
    fig.show()



# plot_hks_graph_subplots(centroids, cell_graph, (cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))/np.std(cell_weighted_hks, axis=0))

curvature_proxy = HKS_times*(cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))
# curvature_proxy = HKS_times*(cell_weighted_hks)

print(np.std(curvature_proxy, axis=0))
z_curvature = curvature_proxy/(np.std(curvature_proxy, axis=0))
plot_hks_graph_subplots(centroids, cell_graph, z_curvature)


[0.00325305 0.00549515 0.00934892 0.01579347 0.02118703 0.02563891
 0.02913898 0.03226907]


In [123]:
import plotly.graph_objects as go

def plot_organoid_loops_3d(coords: np.ndarray,
                           G: nx.Graph,
                           z_small: np.ndarray,
                           loops: list[np.ndarray],
                           title: str = "") -> go.Figure:
    """
    Minimal 3D plot:
      - nodes colored by z_small (RdBu_r) with symmetric limits around 0
      - light wireframe edges
      - each loop as a thick polyline in a distinct color
    """
    XYZ = coords[:, :3]

    # wireframe
    ex, ey, ez = [], [], []
    for u, v in G.edges():
        ex += [XYZ[u,0], XYZ[v,0], None]
        ey += [XYZ[u,1], XYZ[v,1], None]
        ez += [XYZ[u,2], XYZ[v,2], None]

    traces = []
    if ex:
        traces.append(go.Scatter3d(
            x=ex, y=ey, z=ez, mode="lines",
            line=dict(width=1.0, color="rgba(120,120,120,0.75)"),
            opacity=1.0, hoverinfo="skip", name="wireframe", showlegend=False
        ))

    # symmetric color limits so 0 is centered
    finite = np.isfinite(z_small)
    vmax = float(np.nanmax(np.abs(z_small[finite]))) if np.any(finite) else 1.0
    if vmax == 0.0:
        vmax = 1.0  # avoid zero range

    # nodes
    traces.append(go.Scatter3d(
        x=XYZ[:,0], y=XYZ[:,1], z=XYZ[:,2],
        mode="markers",
        marker=dict(
            size=3.5,
            color=z_small,
            colorscale="RdBu_r",
            showscale=True,
            colorbar=dict(title="z-HKS (small t)"),
            opacity=0.95,
            cmin=-vmax, cmax=vmax, cmid=0.0
        ),
        hoverinfo="skip", name="cells", showlegend=False
    ))

    # loops
    palette = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
               "#ff7f00", "#a65628", "#f781bf", "#999999"]
    for i, poly in enumerate(loops):
        if poly.size < 2:
            continue
        # ensure closed
        if poly[0] != poly[-1]:
            poly = np.concatenate([poly, poly[:1]])
        traces.append(go.Scatter3d(
            x=XYZ[poly,0], y=XYZ[poly,1], z=XYZ[poly,2],
            mode="lines",
            line=dict(width=7.0, color=palette[i % len(palette)]),
            hoverinfo="skip", name=f"loop {i+1}", showlegend=True
        ))

    mins = XYZ.min(0); maxs = XYZ.max(0); span = float(np.max(maxs - mins)); ctr = (mins + maxs)/2
    fig = go.Figure(traces)
    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[ctr[0]-span/2, ctr[0]+span/2], showgrid=False, showticklabels=False, zeroline=False),
            yaxis=dict(range=[ctr[1]-span/2, ctr[1]+span/2], showgrid=False, showticklabels=False, zeroline=False),
            zaxis=dict(range=[ctr[2]-span/2, ctr[2]+span/2], showgrid=False, showticklabels=False, zeroline=False),
            aspectmode="cube",
        ),
        margin=dict(l=0,r=0,t=30,b=0),
        title=title,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0)
    )
    return fig

In [124]:
from __future__ import annotations
import numpy as np
import networkx as nx
import plotly.graph_objects as go

# ---------------------------
# Step 1 — high-HKS mask (all time-scales)
# ---------------------------

def make_high_mask(HKS, thr_allpos):
    """
    Binary mask of nodes that satisfy:
      - HKS at ALL time-scales (columns) >= thr_allpos
    """
    if HKS.ndim != 2 or HKS.shape[1] < 1:
        raise ValueError("HKS must be a 2D array with at least 1 time-scale (column).")
    return np.all(HKS >= thr_allpos, axis=1)


def find_high_regions(G, high_mask, min_region_size=5):
    """
    Find connected components of the subgraph induced by 'high_mask'.
    Keep only components with at least 'min_region_size' nodes.
    Each returned region is a set of node indices (connected patch).
    """
    if G.number_of_nodes() != high_mask.shape[0]:
        raise ValueError("G and high_mask size mismatch.")
    nodes_high = [n for n in G.nodes if high_mask[n]]
    sub = G.subgraph(nodes_high)
    comps = [set(c) for c in nx.connected_components(sub)]
    comps = [c for c in comps if len(c) >= min_region_size]
    return comps


# ---------------------------
# Debug visualization helper (overlays for crypt regions)
# ---------------------------

def add_crypt_overlays(fig, coords, regions):
    """
    Overlay translucent colored markers for each crypt region.
    """
    XYZ = coords[:, :3]
    region_colors = [
        "rgba(227, 26, 28, 0.25)",  # red
        "rgba(31, 120, 180, 0.25)", # blue
        "rgba(51, 160, 44, 0.25)",  # green
        "rgba(106, 61, 154, 0.25)", # purple
        "rgba(255, 127, 0, 0.25)",  # orange
        "rgba(166, 86, 40, 0.25)",  # brown
        "rgba(152, 78, 163, 0.25)", # violet
        "rgba(0, 0, 0, 0.25)",      # gray
    ]
    for k, reg in enumerate(regions):
        idx = np.fromiter(reg, dtype=int)
        if idx.size == 0:
            continue
        fig.add_trace(go.Scatter3d(
            x=XYZ[idx, 0], y=XYZ[idx, 1], z=XYZ[idx, 2],
            mode="markers",
            marker=dict(size=6.5, color=region_colors[k % len(region_colors)]),
            name=f"crypt {k+1}",
            showlegend=True,
            hoverinfo="skip",
        ))


import numpy as np
import networkx as nx

def _false_components(G, high_mask):
    """Connected components induced by nodes where high_mask == False."""
    false_nodes = [n for n in G.nodes if not high_mask[n]]
    sub = G.subgraph(false_nodes)
    return [set(c) for c in nx.connected_components(sub)]

def _is_enclosed(G, comp, high_mask):
    """Check if every neighbor just outside 'comp' is True in high_mask."""
    boundary_neighbors = set()
    for u in comp:
        for v in G.neighbors(u):
            if v not in comp:
                boundary_neighbors.add(v)
    if not boundary_neighbors:
        # Isolated pocket with no outside neighbors (unlikely); treat as enclosed.
        return True
    # Enclosed iff all boundary neighbors are True
    return all(high_mask[v] for v in boundary_neighbors)

def fill_small_crypt_defects(G, high_mask, max_hole_size=5, verbose=True):
    """
    Fill small 'defect' patches (False) that are completely surrounded by True nodes.
    A defect is filled if:
      - it is a connected component of False nodes of size <= max_hole_size, and
      - ALL its boundary neighbors (outside the component) are True.

    Parameters
    ----------
    G : networkx.Graph
        Graph with N nodes labeled 0..N-1.
    high_mask : np.ndarray (bool, shape (N,))
        True for nodes that meet the crypt threshold across all time-scales.
    max_hole_size : int
        Maximum size of a defect component to fill (default 5).
    verbose : bool
        Print how many patches were filled.

    Returns
    -------
    high_mask_filled : np.ndarray (bool, shape (N,))
        Updated mask with small enclosed defects flipped to True.
    filled_patches : list of sets
        Each set is the node indices of a filled defect component.
    """
    if high_mask.dtype != bool:
        raise ValueError("high_mask must be a boolean array.")
    if G.number_of_nodes() != high_mask.shape[0]:
        raise ValueError("G and high_mask size mismatch.")

    high_mask_filled = high_mask.copy()
    filled_patches = []

    comps = _false_components(G, high_mask_filled)
    for comp in comps:
        if len(comp) <= max_hole_size and _is_enclosed(G, comp, high_mask_filled):
            # Fill this defect: flip to True
            idx = np.fromiter(comp, dtype=int)
            high_mask_filled[idx] = True
            filled_patches.append(comp)

    if verbose:
        print(f"Filled {len(filled_patches)} enclosed defect patch(es) (size ≤ {max_hole_size}).")

    return high_mask_filled, filled_patches


import numpy as np
import networkx as nx

# ---------------------------
# Boundary & loop extraction
# ---------------------------

def region_boundary_nodes(G, region):
    """Nodes of 'region' that touch at least one neighbor outside the region."""
    boundary = set()
    region_set = set(region)
    for u in region:
        for v in G.neighbors(u):
            if v not in region_set:
                boundary.add(u)
                break
    return boundary

def boundary_subgraph(G, boundary_nodes):
    """Subgraph induced by boundary nodes."""
    return G.subgraph(boundary_nodes).copy()

def extract_loops_on_boundary(B, min_loop_len=5):
    """
    Use cycle basis on the boundary subgraph.
    Keep cycles of length >= min_loop_len.
    """
    if B.number_of_nodes() == 0:
        return []
    loops = nx.cycle_basis(B)
    loops = [np.asarray(c, dtype=int) for c in loops if len(c) >= min_loop_len]
    return loops

def initial_boundary_loops_for_region(G, region, min_loop_len=5):
    """Convenience: get initial boundary loops for one region."""
    bnodes = region_boundary_nodes(G, region)
    B = boundary_subgraph(G, bnodes)
    return extract_loops_on_boundary(B, min_loop_len=min_loop_len)

# ---------------------------
# Loop utilities
# ---------------------------

def loop_cost(z_small, loop_nodes):
    """Sum of z_small along the loop (lower is better)."""
    return float(np.sum(z_small[np.asarray(loop_nodes, dtype=int)]))

def _neighbors_that_link_prev_next(G, prev_n, next_n):
    """
    Return neighbors 'w' that are adjacent to BOTH prev_n and next_n
    (i.e., allow replacing the middle node with 'w' while keeping a simple cycle).
    Includes possibility that prev_n and/or next_n themselves appear, but those
    will be filtered by caller.
    """
    # intersect neighbor sets
    n_prev = set(G.neighbors(prev_n))
    n_next = set(G.neighbors(next_n))
    return n_prev & n_next

def _tighten_once(G, loop_nodes, z_small, forbidden, allow_remove=True):
    """
    One greedy coordinate-descent step over the loop.
    Tries either:
      - removal of a node v if prev-next edge exists (shrinks length), or
      - replacement of v by a neighbor w that connects to both prev and next and lowers cost.
    Constraints:
      - 'forbidden' nodes (crypt region) cannot be part of the loop.
      - Loop length must not increase (removal decreases; replacement keeps same).
      - Replacement 'w' must not already be in the loop (except prev/next).
    Returns (new_loop_nodes, improved: bool).
    """
    L = len(loop_nodes)
    if L < 3:
        return loop_nodes, False

    loop = list(loop_nodes)
    loop_set = set(loop)

    best_delta = 0.0
    best_action = None  # ("remove", idx) or ("replace", idx, w)
    current_cost = loop_cost(z_small, loop)

    for i in range(L):
        prev_i = (i - 1) % L
        next_i = (i + 1) % L
        v = loop[i]
        p = loop[prev_i]
        n = loop[next_i]

        # --- Option A: removal (shrink) if prev and next are adjacent
        if allow_remove and G.has_edge(p, n):
            # New loop would be loop without v
            # Ensure we don't bring in forbidden nodes (we don't add any node).
            # Cost delta: - z_small[v]
            delta = -float(z_small[v])
            if delta < best_delta:  # we minimize cost (delta negative is better)
                best_delta = delta
                best_action = ("remove", i)

        # --- Option B: replacement (slide) to a neighbor w that links p and n
        candidates = _neighbors_that_link_prev_next(G, p, n)
        for w in candidates:
            if w == v or w == p or w == n:
                continue
            if w in forbidden:
                continue
            if w in loop_set:
                continue  # avoid self-intersections / reuse
            # Cost delta if we replace v with w (same length)
            delta = float(z_small[w] - z_small[v])
            if delta < best_delta:
                best_delta = delta
                best_action = ("replace", i, w)

    if best_action is None:
        return loop_nodes, False

    # Apply the best local action
    if best_action[0] == "remove":
        i = best_action[1]
        new_loop = loop[:i] + loop[i+1:]
        return np.asarray(new_loop, dtype=int), True
    else:
        _, i, w = best_action
        new_loop = loop.copy()
        new_loop[i] = w
        return np.asarray(new_loop, dtype=int), True

def tighten_loop(G, loop_nodes, z_small, forbidden, min_loop_len=5, max_iters=200):
    """
    Iteratively tighten a loop by greedy local moves that never increase length and
    (ideally) lower the total z_small cost. Stops when no improvement occurs.
    - 'forbidden' are crypt nodes: loop may not enter them.
    - If loop shrinks below 'min_loop_len', it is discarded (returns None).
    """
    loop = np.asarray(loop_nodes, dtype=int)
    if loop.size < min_loop_len:
        return None

    prev_cost = loop_cost(z_small, loop)
    for _ in range(max_iters):
        loop_new, improved = _tighten_once(G, loop, z_small, forbidden, allow_remove=True)
        if not improved:
            break
        if loop_new.size < min_loop_len:
            return None
        # If length same but cost increased (shouldn’t happen with our selection), guard:
        new_cost = loop_cost(z_small, loop_new)
        if loop_new.size > loop.size:
            # Should never happen with our moves, but enforce non-increase:
            break
        loop = loop_new
        prev_cost = new_cost

    # Final pass without allowing removals, to settle geometrically without over-shrinking
    for _ in range(50):
        loop_new, improved = _tighten_once(G, loop, z_small, forbidden, allow_remove=False)
        if not improved:
            break
        if loop_new.size < min_loop_len:
            return None
        loop = loop_new

    return np.asarray(loop, dtype=int)

# ---------------------------
# Orchestrator (updated)
# ---------------------------

def detect_crypt_patches(
    coords,
    G,
    HKS,
    thr_allpos,
    min_region_size=5,
    plot_func=None,
    title="Crypt detection (debug)"
):
    """
    Identify crypt-like patches (HKS >= thr_allpos at all time-scales), fill small defects,
    then extract boundary loops and tighten them away from the crypt to lower HKS (t=small),
    never increasing loop length. Plots crypt regions and final loops.
    """
    # Step 1: thresholding across all time-scales
    high_mask = make_high_mask(HKS, thr_allpos=thr_allpos)

    # Fill tiny enclosed defects (≤5 nodes)
    high_mask_filled, _ = fill_small_crypt_defects(G, high_mask, max_hole_size=10)

    # Step 2: connected crypt regions
    regions = find_high_regions(G, high_mask_filled, min_region_size=min_region_size)
    print(f"Found {len(regions)} crypt region(s).")
    for i, reg in enumerate(regions, 1):
        print(f"  - Crypt {i}: {len(reg)} nodes")

    # ---------------------------
    # Step 3: initial boundary loops per region
    # ---------------------------
    initial_loops_by_region = []
    for reg in regions:
        loops0 = initial_boundary_loops_for_region(G, reg, min_loop_len=5)
        initial_loops_by_region.append(loops0)

    initial_loops = [lp for loops0 in initial_loops_by_region for lp in loops0]


    # ---------------------------
    # Step 4: tighten loops away from crypt (toward lower HKS at small t)
    # ---------------------------
    z_small = HKS[:, 0]
    final_loops = []
    for reg, loops0 in zip(regions, initial_loops_by_region):
        forb = set(reg)  # crypt cells are forbidden for the neck loop
        for loop0 in loops0:
            loop_final = tighten_loop(G, loop0, z_small, forbidden=forb, min_loop_len=5, max_iters=200)
            if loop_final is not None:
                final_loops.append(loop_final)

    print(f"Tightened to {len(final_loops)} loop(s) total.")

    # Optional plotting
    fig = None
    if plot_func is not None:
        fig = plot_func(coords=coords, G=G, z_small=z_small, loops=initial_loops, title=title)
        add_crypt_overlays(fig, coords, regions)

    return {
        "high_mask": high_mask,
        "regions": regions,
        "loops": final_loops,
        "figure": fig
    }



HKS_zscored = (cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))/np.std(cell_weighted_hks, axis=0)


result = detect_crypt_patches(
    coords=centroids,
    G=cell_graph,
    HKS=HKS_zscored[:,:6],
    thr_allpos=0.1,     # example: require HKS >= 0.15 for every time-scale
    min_region_size=5,
    plot_func=plot_organoid_loops_3d,
    title="Crypt patches (debug)"
)

fig = result["figure"]
if fig is not None:
    fig.show()





Filled 0 enclosed defect patch(es) (size ≤ 10).
Found 5 crypt region(s).
  - Crypt 1: 59 nodes
  - Crypt 2: 46 nodes
  - Crypt 3: 21 nodes
  - Crypt 4: 25 nodes
  - Crypt 5: 43 nodes
Tightened to 5 loop(s) total.


In [125]:
import numpy as np
import networkx as nx



# ---------------------------
# Masks (all time-scales)
# ---------------------------

def make_high_mask(HKS, thr_allpos):
    """
    True where every time-scale is >= thr_allpos.
    """
    if HKS.ndim != 2 or HKS.shape[1] < 1:
        raise ValueError("HKS must be a 2D array with at least 1 column.")
    return np.all(HKS >= thr_allpos, axis=1)

def make_low_mask(HKS, thr_allneg):
    """
    True where every time-scale is <= thr_allneg.
    """
    if HKS.ndim != 2 or HKS.shape[1] < 1:
        raise ValueError("HKS must be a 2D array with at least 1 column.")
    return np.all(HKS <= thr_allneg, axis=1)

# ---------------------------
# Connected regions (seeds)
# ---------------------------

def find_regions_from_mask(G, mask, min_region_size=5):
    """
    Connected components induced by nodes where mask is True.
    Keep only components with at least min_region_size nodes.
    """
    if G.number_of_nodes() != mask.shape[0]:
        raise ValueError("G and mask size mismatch.")
    nodes = [n for n in G.nodes if mask[n]]
    sub = G.subgraph(nodes)
    comps = [set(c) for c in nx.connected_components(sub)]
    return [c for c in comps if len(c) >= min_region_size]

# ---------------------------
# Overlays for visualization
# ---------------------------

def add_crypt_overlays(fig, coords, regions):
    """
    Translucent overlays for crypt seed regions (warm palette).
    """
    XYZ = coords[:, :3]
    colors = [
        "rgba(227, 26, 28, 0.32)",  # red
        "rgba(255, 127, 0, 0.32)",  # orange
        "rgba(166, 86, 40, 0.32)",  # brown
        "rgba(231, 41, 138, 0.32)", # magenta
        "rgba(252, 141, 98, 0.32)", # salmon
        "rgba(251, 180, 174, 0.32)",# pink
    ]
    for k, reg in enumerate(regions):
        idx = np.fromiter(reg, dtype=int)
        if idx.size == 0: continue
        fig.add_trace(go.Scatter3d(
            x=XYZ[idx,0], y=XYZ[idx,1], z=XYZ[idx,2],
            mode="markers",
            marker=dict(size=6.5, color=colors[k % len(colors)]),
            name=f"crypt seed {k+1}",
            showlegend=True, hoverinfo="skip"
        ))

def add_neck_overlays(fig, coords, regions):
    """
    Translucent overlays for neck seed regions (cool palette).
    """
    XYZ = coords[:, :3]
    colors = [
        "rgba(31, 120, 180, 0.36)",  # blue
        "rgba(166, 206, 227, 0.36)", # light blue
        "rgba(57, 146, 131, 0.36)",  # teal
        "rgba(107, 174, 214, 0.36)", # sky
        "rgba(44, 123, 182, 0.36)",  # steel
        "rgba(158, 202, 225, 0.36)", # pale blue
    ]
    for k, reg in enumerate(regions):
        idx = np.fromiter(reg, dtype=int)
        if idx.size == 0: continue
        fig.add_trace(go.Scatter3d(
            x=XYZ[idx,0], y=XYZ[idx,1], z=XYZ[idx,2],
            mode="markers",
            marker=dict(size=7.5, color=colors[k % len(colors)]),
            name=f"neck seed {k+1}",
            showlegend=True, hoverinfo="skip"
        ))


# ---------------------------
# Helpers: labeling utilities
# ---------------------------

def init_patch_labels(n_nodes, regions):
    """
    Create a label array for nodes from initial regions.
    labels[i] = -1 means unassigned; otherwise it's the patch id (0..K-1).
    Also returns a list of sets for fast membership per patch id.
    """
    labels = -1 * np.ones(n_nodes, dtype=int)
    patch_sets = []
    for pid, reg in enumerate(regions):
        s = set(reg)
        patch_sets.append(s)
        for u in s:
            labels[u] = pid
    return labels, patch_sets


def frontier_nodes(G, patch_nodes, labels):
    """
    Nodes adjacent to 'patch_nodes' that are currently unlabeled (labels == -1).
    Returns a Python set.
    """
    cand = set()
    for u in patch_nodes:
        for v in G.neighbors(u):
            if labels[v] == -1:
                cand.add(v)
    return cand


def count_neighbors_in_set(G, node, node_set):
    """
    Count how many neighbors of 'node' lie in 'node_set'.
    """
    c = 0
    for v in G.neighbors(node):
        if v in node_set:
            c += 1
    return c


# ---------------------------
# One-step growth for a patch
# ---------------------------

def grow_one_step_for_patch(G, patch_set, labels, HKS_t0, thr_grow_t0):
    """
    Try to grow 'patch_set' by adding exactly one node, following rules:
      2a) HKS_t0[node] >= thr_grow_t0
      2b) node has at least 2 neighbors already inside patch_set
    Candidates are frontier nodes (unlabeled neighbors), sorted by HKS_t0 desc.
    Returns (added_node or None).
    """
    cand = list(frontier_nodes(G, patch_set, labels))
    if not cand:
        return None

    # Sort by descending HKS at t0
    cand.sort(key=lambda u: HKS_t0[u], reverse=True)

    for u in cand:
        if HKS_t0[u] >= thr_grow_t0 and count_neighbors_in_set(G, u, patch_set) >= 3:
            # Add this node to the patch
            patch_set.add(u)
            labels[u] = labels[next(iter(patch_set))]  # use any member's label (constant per set)
            return u

    return None


# ---------------------------
# Merge touching patches
# ---------------------------

def merge_touching_patches(G, labels, patch_sets):
    """
    If two different labeled patches touch (there exists an edge across labels),
    merge them into a single patch. Returns (merged, labels, patch_sets).
    'merged' is True iff any merge happened.
    """
    # Build unions of touching labels
    parent = {pid: pid for pid in range(len(patch_sets))}

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # For each edge, if endpoints have different non-negative labels, union them
    for u, v in G.edges():
        lu, lv = labels[u], labels[v]
        if lu >= 0 and lv >= 0 and lu != lv:
            union(lu, lv)

    # Build mapping to representatives
    reps = {pid: find(pid) for pid in range(len(patch_sets))}
    # If no merges, early exit
    if all(pid == reps[pid] for pid in reps):
        return False, labels, patch_sets

    # Rebuild merged patch sets
    new_id_map = {}
    new_sets = []
    for old_id, rep in reps.items():
        if old_id not in patch_sets:
            continue
    # Pack all old ids by representative
    groups = {}
    for old_id, rep in reps.items():
        groups.setdefault(rep, []).append(old_id)

    # Create new sets and id map
    for new_pid, rep in enumerate(groups.keys()):
        merged_set = set()
        for old_id in groups[rep]:
            merged_set |= patch_sets[old_id]
        new_sets.append(merged_set)
        for old_id in groups[rep]:
            new_id_map[old_id] = new_pid

    # Relabel labels according to new ids
    new_labels = -1 * np.ones_like(labels)
    for new_pid, s in enumerate(new_sets):
        for u in s:
            new_labels[u] = new_pid

    return True, new_labels, new_sets


# --- New helper: add exactly one node to a specific patch id ---

def grow_one_step_for_patch_with_pid(G, patch_sets, pid, labels, HKS_t0, thr_grow_t0):
    """
    Try to add exactly one node to patch 'pid':
      - Candidate must be unlabeled, adjacent to the patch,
      - HKS_t0 >= thr_grow_t0,
      - Has at least 2 neighbors inside the patch.
    Returns the added node id, or None if no candidate fits.
    """
    patch = patch_sets[pid]
    cand = list(frontier_nodes(G, patch, labels))
    if not cand:
        return None

    cand.sort(key=lambda u: HKS_t0[u], reverse=True)

    for u in cand:
        if HKS_t0[u] >= thr_grow_t0 and count_neighbors_in_set(G, u, patch) >= 3:
            patch.add(u)
            labels[u] = pid
            return u

    return None


# ---------------------------
# Orchestrated multi-sweep growth
# ---------------------------

def grow_patches_until_stable(G, regions, HKS, thr_grow_t0, max_sweeps=1000, verbose=True):
    """
    Multi-source region growing with your rules:
      - For each patch, try to add exactly one best frontier node per sweep
      - Node must satisfy HKS_t0 >= thr_grow_t0 and have >=2 neighbors in the patch
      - After each sweep, merge any touching patches
      - Repeat until a full sweep adds nothing and no merges occur

    Returns (labels, patch_sets), where patch_sets is a list of sets of node indices.
    """
    n = G.number_of_nodes()
    HKS_t0 = HKS[:, 0]

    labels, patch_sets = init_patch_labels(n, regions)

    for sweep in range(max_sweeps):
        any_added = False

        # One attempt per patch (fixed order over current patch ids)
        for pid in range(len(patch_sets)):
            patch = patch_sets[pid]
            if not patch:
                continue
            # Grow exactly one node if possible
            added = grow_one_step_for_patch(G, patch, labels, HKS_t0, thr_grow_t0)
            if added is not None:
                any_added = True

        # Merge any patches that now touch
        merged, labels, patch_sets = merge_touching_patches(G, labels, patch_sets)

        if verbose:
            total_size = sum(len(s) for s in patch_sets)
            print(f"Sweep {sweep+1}: added={any_added}, merged={merged}, "
                  f"num_patches={len(patch_sets)}, total_nodes_in_patches={total_size}")

        if not any_added and not merged:
            if verbose:
                print("Growth stabilized.")
            break

    return labels, patch_sets


# --- Alternative growth orchestrator: patch-outer, then sweep per patch ---

def grow_patches_until_stable_patch_outer(
    G,
    regions,
    HKS,
    thr_grow_t0,
    max_outer_passes=100,
    max_inner_steps_per_patch=None,
    verbose=True
):
    """
    Patch-outer growth:
      - Initialize labels/sets from 'regions'.
      - For pid in patches:
          * Keep adding nodes to this patch (highest HKS_t0 first) until it stalls
            (or 'max_inner_steps_per_patch' is reached).
          * Merge touching patches.
          * If a merge happened, restart from pid = 0 (indices changed).
      - Repeat outer passes until an entire pass yields no changes (no adds, no merges).

    Returns (labels, patch_sets).
    """
    n = G.number_of_nodes()
    HKS_t0 = HKS[:, 0]

    labels, patch_sets = init_patch_labels(n, regions)

    for outer in range(max_outer_passes):
        any_added_in_pass = False
        any_merged_in_pass = False

        pid = 0
        while pid < len(patch_sets):
            patch = patch_sets[pid]
            if not patch:
                pid += 1
                continue

            added_this_patch = 0
            # Inner sweep: keep growing this patch until it stalls
            while True:
                added = grow_one_step_for_patch_with_pid(
                    G=G,
                    patch_sets=patch_sets,
                    pid=pid,
                    labels=labels,
                    HKS_t0=HKS_t0,
                    thr_grow_t0=thr_grow_t0
                )
                if added is None:
                    break
                added_this_patch += 1
                any_added_in_pass = True
                if max_inner_steps_per_patch is not None and added_this_patch >= max_inner_steps_per_patch:
                    break

            # Merge after finishing this patch’s inner sweep
            merged, labels, patch_sets = merge_touching_patches(G, labels, patch_sets)
            if merged:
                any_merged_in_pass = True
                # Patches have been reindexed -> restart from the beginning
                pid = 0
                if verbose:
                    print(f"[outer {outer+1}] merged; restarting patch loop. "
                          f"num_patches={len(patch_sets)}")
                continue

            if verbose:
                print(f"[outer {outer+1}] patch {pid}: added {added_this_patch} node(s); "
                      f"size={len(patch_sets[pid])}")

            pid += 1

        if verbose:
            total_size = sum(len(s) for s in patch_sets)
            print(f"== End outer pass {outer+1}: any_added={any_added_in_pass}, "
                  f"any_merged={any_merged_in_pass}, num_patches={len(patch_sets)}, "
                  f"total_nodes_in_patches={total_size}")

        if not any_added_in_pass and not any_merged_in_pass:
            if verbose:
                print("Growth stabilized (patch-outer).")
            break

    # Diagnostics for each final patch
    stall_reports = []
    HKS_t0 = HKS[:, 0]
    for pid, patch in enumerate(patch_sets):
        rpt = diagnose_patch_stall(G, patch, labels, HKS_t0, thr_grow_t0)
        pretty_print_stall_report(pid, rpt)
        stall_reports.append(rpt)

    return labels, patch_sets, stall_reports



def get_frontier_candidates_sorted(G, patch_set, labels, HKS_t0):
    """
    Return a list of unlabeled frontier nodes sorted by descending HKS_t0.
    """
    cand = list(frontier_nodes(G, patch_set, labels))
    cand.sort(key=lambda u: HKS_t0[u], reverse=True)
    return cand

def diagnose_patch_stall(G, patch_set, labels, HKS_t0, thr_grow_t0):
    """
    Diagnose why this patch cannot grow further given the current labels.
    Returns a dict with reason, details, and the 'next best' candidate that failed.
    """
    cand = get_frontier_candidates_sorted(G, patch_set, labels, HKS_t0)

    report = {
        "reason": None,
        "details": "",
        "next_node": None,
        "next_node_hks": None,
        "next_node_neighbors_in_patch": None,
        "num_frontier": len(cand),
        "num_frontier_above_thr": 0,
        "best_frontier_hks": None,
    }

    if len(cand) == 0:
        report["reason"] = "no_frontier"
        report["details"] = "No unlabeled neighbors adjacent to the patch."
        return report

    # Best overall HKS among frontier
    best = cand[0]
    report["best_frontier_hks"] = float(HKS_t0[best])

    # Split by threshold
    above = [u for u in cand if HKS_t0[u] >= thr_grow_t0]
    report["num_frontier_above_thr"] = len(above)

    if len(above) == 0:
        # Nobody passes the HKS threshold: pick the best to report
        report["reason"] = "below_threshold"
        report["next_node"] = int(best)
        report["next_node_hks"] = float(HKS_t0[best])
        report["next_node_neighbors_in_patch"] = int(count_neighbors_in_set(G, best, patch_set))
        report["details"] = (
            f"No frontier nodes meet HKS threshold (thr_grow_t0={thr_grow_t0}). "
            f"Best frontier node {report['next_node']} has HKS={report['next_node_hks']:.4f}."
        )
        return report

    # At least one node is above threshold, but growth still stalled -> all fail the 2-neighbor rule
    # Report the best-above-threshold node
    top_ok_hks = above[0]
    nn = count_neighbors_in_set(G, top_ok_hks, patch_set)
    report["reason"] = "insufficient_connections"
    report["next_node"] = int(top_ok_hks)
    report["next_node_hks"] = float(HKS_t0[top_ok_hks])
    report["next_node_neighbors_in_patch"] = int(nn)
    report["details"] = (
        "Frontier nodes above threshold exist but none have ≥2 neighbors in the patch. "
        f"Best candidate {report['next_node']} has HKS={report['next_node_hks']:.4f} "
        f"and {nn} neighbor(s) in the patch."
    )
    return report

def pretty_print_stall_report(patch_id, report):
    """
    Human-friendly one-liner (plus context) for logs.
    """
    reason = report["reason"]
    if reason == "no_frontier":
        msg = (f"Patch {patch_id}: stopped — no frontier neighbors. "
               f"(num_frontier=0)")
    elif reason == "below_threshold":
        msg = (f"Patch {patch_id}: stopped — no frontier nodes meet HKS threshold. "
               f"Best frontier node {report['next_node']} HKS={report['next_node_hks']:.4f} "
               f"(neighbors_in_patch={report['next_node_neighbors_in_patch']}).")
    elif reason == "insufficient_connections":
        msg = (f"Patch {patch_id}: stopped — insufficient connections (need ≥2). "
               f"Best above-threshold node {report['next_node']} "
               f"HKS={report['next_node_hks']:.4f}, "
               f"neighbors_in_patch={report['next_node_neighbors_in_patch']}. "
               f"(frontier_above_thr={report['num_frontier_above_thr']})")
    else:
        msg = f"Patch {patch_id}: stopped — reason unknown."
    print(msg)
    return msg



def grow_one_step_for_neck_with_pid(
    G,
    patch_sets,
    pid,
    labels,
    HKS_t0,
    thr_neck_t0,
    min_neighbors_in_patch=2
):
    """
    Try to add exactly one node to NECK patch 'pid'.
    Rules:
      - Candidate must be unlabeled and adjacent to the patch,
      - HKS_t0[cand] <= thr_neck_t0  (opposite of crypt growth),
      - Candidate has at least 'min_neighbors_in_patch' neighbors already in the patch.
    Tie-breaking:
      - Sort candidates by ascending HKS_t0 (lowest first), then add the first that fits.

    Returns:
      added_node (int) or None if no candidate fits.
    """
    patch = patch_sets[pid]
    cand = list(frontier_nodes(G, patch, labels))
    if not cand:
        return None

    # For necks, lower HKS is "better": sort ascending
    cand.sort(key=lambda u: HKS_t0[u])

    for u in cand:
        if HKS_t0[u] <= thr_neck_t0 and count_neighbors_in_set(G, u, patch) >= min_neighbors_in_patch:
            patch.add(u)
            labels[u] = pid
            return u

    return None


# ---------------------------------
# Orchestrator: patch-outer growth for necks
# ---------------------------------

def grow_neck_patches_until_stable_patch_outer(
    G,
    regions,               # initial neck seed regions (sets of nodes)
    HKS,                   # (N, T) HKS array
    thr_neck_t0,           # growth threshold on the lowest time-scale
    min_neighbors_in_patch=2,
    max_outer_passes=100,
    max_inner_steps_per_patch=None,  # optional cap per patch per outer pass
    verbose=True
):
    """
    Patch-outer growth for NECK regions:
      - Initialize from 'regions'.
      - For each patch id in order:
          * Keep adding nodes (lowest HKS_t0 first) while:
              HKS_t0 <= thr_neck_t0 and neighbors_in_patch >= min_neighbors_in_patch.
          * After finishing the patch, merge touching patches.
          * If a merge occurs, restart from the first patch (indices change).
      - Repeat outer passes until a full pass yields no adds and no merges.

    Returns:
      labels (np.ndarray, shape (N,)), patch_sets (list of sets of node ids)
    """
    n = G.number_of_nodes()
    HKS_t0 = HKS[:, 0]
    labels, patch_sets = init_patch_labels(n, regions)

    for outer in range(max_outer_passes):
        any_added_in_pass = False
        any_merged_in_pass = False

        pid = 0
        while pid < len(patch_sets):
            patch = patch_sets[pid]
            if not patch:
                pid += 1
                continue

            added_this_patch = 0
            while True:
                added = grow_one_step_for_neck_with_pid(
                    G=G,
                    patch_sets=patch_sets,
                    pid=pid,
                    labels=labels,
                    HKS_t0=HKS_t0,
                    thr_neck_t0=thr_neck_t0,
                    min_neighbors_in_patch=min_neighbors_in_patch
                )
                if added is None:
                    break
                added_this_patch += 1
                any_added_in_pass = True
                if max_inner_steps_per_patch is not None and added_this_patch >= max_inner_steps_per_patch:
                    break

            merged, labels, patch_sets = merge_touching_patches(G, labels, patch_sets)
            if merged:
                any_merged_in_pass = True
                pid = 0  # restart due to reindexing
                if verbose:
                    print(f"[neck outer {outer+1}] merged; restart. num_patches={len(patch_sets)}")
                continue

            if verbose:
                print(f"[neck outer {outer+1}] patch {pid}: +{added_this_patch} nodes; size={len(patch_sets[pid])}")
            pid += 1

        if verbose:
            total_size = sum(len(s) for s in patch_sets)
            print(f"== End neck outer pass {outer+1}: any_added={any_added_in_pass}, "
                  f"any_merged={any_merged_in_pass}, patches={len(patch_sets)}, "
                  f"nodes_in_patches={total_size}")

        if not any_added_in_pass and not any_merged_in_pass:
            if verbose:
                print("Neck growth stabilized (patch-outer).")
            break

    return labels, patch_sets



def detect_crypt_patches(
    coords,
    G,
    HKS,
    thr_allpos,
    min_region_size=5,
    plot_func=None,
    title="Crypt detection (debug)",
    # NEW: growth threshold on the lowest time-scale
    thr_grow_t0=None,
    do_growth=True
):
    """
    Identify crypt-like patches (all-scales threshold), fill tiny defects,
    connect components, and (optionally) grow patches outward using rules 2a/2b.
    """
    # Step 1: thresholding across all time-scales
    high_mask = make_high_mask(HKS, thr_allpos=thr_allpos)

    # Fill tiny enclosed defects (≤5)
    high_mask_filled, _filled = fill_small_crypt_defects(G, high_mask, max_hole_size=5)

    # Step 2: connected crypt regions
    regions = find_high_regions(G, high_mask_filled, min_region_size=min_region_size)
    print(f"Found {len(regions)} crypt region(s).")
    for i, reg in enumerate(regions, 1):
        print(f"  - Crypt {i}: {len(reg)} nodes")

    grown_labels = None
    grown_patches = None

    # ---------------------------
    # NEW: Growth stage (your algorithm)
    # ---------------------------
    if do_growth:
        if thr_grow_t0 is None:
            thr_grow_t0 = thr_allpos  # sensible default: same as all-scales positivity
        # grown_labels, grown_patches = grow_patches_until_stable(
        #     G=G,
        #     regions=regions,
        #     HKS=HKS,
        #     thr_grow_t0=thr_grow_t0,
        #     max_sweeps=1000,
        #     verbose=True
        # )

        grown_labels, grown_patches, stall_reports = grow_patches_until_stable_patch_outer(
            G=G,
            regions=regions,           # initial crypt patches after filling
            HKS=HKS,
            thr_grow_t0=thr_grow_t0,   # e.g., same as thr_allpos or a tad lower
            max_outer_passes=100000,
            max_inner_steps_per_patch=None,  # or set a cap if you want stricter control
            verbose=True
        )


        print(f"After growth: {len(grown_patches)} patch(es).")
        for i, reg in enumerate(grown_patches, 1):
            print(f"  - Grown patch {i}: {len(reg)} nodes")

    # Optional plotting (same as before)
    fig = None
    if plot_func is not None:
        z_small = HKS[:, 0]
        fig = plot_func(coords=coords, G=G, z_small=z_small, loops=[], title=title)
        # Overlay initial crypt regions
        add_crypt_overlays(fig, coords, grown_patches)
        # (Optional) Add a second overlay for grown patches with another color map if desired.

    return {
        "high_mask": high_mask,
        "regions": regions,               # initial crypts after defect fill
        "grown_labels": grown_labels,     # None if do_growth=False
        "grown_patches": grown_patches,   # None if do_growth=False
        "figure": fig
    }



def detect_seeds_crypt_and_neck(
    coords,
    G,
    HKS,
    thr_allpos,          # crypt seed threshold (all time-scales >=)
    thr_allneg,          # neck seed threshold (all time-scales <=)
    min_region_size=5,
    plot_func=None,
    title="Crypt & Neck seeds (debug)"
):
    """
    Identify seed regions for crypts and necks based on all-time-scale thresholds,
    then plot them (no growing yet).

    Returns a dict with masks, regions, and (optional) figure.
    """
    # Masks
    high_mask_all = make_high_mask(HKS, thr_allpos=thr_allpos)     # crypt seeds
    low_mask_all  = make_low_mask(HKS, thr_allneg=thr_allneg)  # neck seeds

    # Regions (connected components)
    crypt_regions = find_regions_from_mask(G, high_mask_all, min_region_size=min_region_size)
    neck_regions  = find_regions_from_mask(G, low_mask_all,  min_region_size=min_region_size)

    print(f"Crypt seeds: {len(crypt_regions)} region(s).")
    for i, reg in enumerate(crypt_regions, 1):
        print(f"  - Crypt seed {i}: {len(reg)} nodes")
    print(f"Neck seeds: {len(neck_regions)} region(s).")
    for i, reg in enumerate(neck_regions, 1):
        print(f"  - Neck seed {i}: {len(reg)} nodes")


    grown_labels, grown_patches, stall_reports = grow_patches_until_stable_patch_outer(
        G=G,
        regions=crypt_regions,           # initial crypt patches after filling
        HKS=HKS,
        thr_grow_t0=-0.2+0.001,   # e.g., same as thr_allpos or a tad lower
        max_outer_passes=100000,
        max_inner_steps_per_patch=None,  # or set a cap if you want stricter control
        verbose=True
    )


    grown_neck_labels, grown_neck_patches = grow_neck_patches_until_stable_patch_outer(
        G=G,
        regions=neck_regions,
        HKS=HKS,                 # can be the sliced early-time HKS if you prefer
        thr_neck_t0=-0.2,       # example: only add nodes with HKS_t0 <= -0.05
        min_neighbors_in_patch=2,
        max_outer_passes=100,
        max_inner_steps_per_patch=None,
        verbose=True
    )
    

    # Plot (nodes colored by lowest-time HKS for context)
    fig = None
    if plot_func is not None:
        z_small = HKS[:, 0]
        fig = plot_func(coords=coords, G=G, z_small=z_small, loops=[], title=title)
        add_crypt_overlays(fig, coords, grown_patches)
        add_neck_overlays(fig, coords, grown_neck_patches)

    return {
        "high_mask_all": high_mask_all,
        "low_mask_all": low_mask_all,
        "crypt_regions": crypt_regions,
        "neck_regions": neck_regions,
        "figure": fig
    }



# HKS_zscored = (cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))/np.std(cell_weighted_hks, axis=0)

curvature_proxy = HKS_times*(cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))
curvature_proxy = curvature_proxy/np.std(curvature_proxy, axis=0)


# result = detect_crypt_patches(
#     coords=centroids,
#     G=cell_graph,
#     HKS=curvature_proxy[:,:4],
#     thr_allpos=0.085,     # example: require HKS >= 0.15 for every time-scale
#     thr_grow_t0= 0.085,
#     min_region_size=5,
#     plot_func=plot_organoid_loops_3d,
#     title="Crypt patches (debug)"
# )

result = detect_seeds_crypt_and_neck(
    coords=centroids,
    G=cell_graph,
    HKS=curvature_proxy[:,:3],
    thr_allpos=0.4,   # crypt seed threshold (all scales >= 0.15)
    thr_allneg=-1.00,  # neck seed threshold (all scales <= -0.05)
    min_region_size=8,
    plot_func=plot_organoid_loops_3d,
    title="Seeds: crypts (warm) & necks (cool)"
)



fig = result["figure"]
if fig is not None:
    fig.show()

Crypt seeds: 5 region(s).
  - Crypt seed 1: 58 nodes
  - Crypt seed 2: 38 nodes
  - Crypt seed 3: 14 nodes
  - Crypt seed 4: 23 nodes
  - Crypt seed 5: 39 nodes
Neck seeds: 2 region(s).
  - Neck seed 1: 24 nodes
  - Neck seed 2: 74 nodes
[outer 1] patch 0: added 12 node(s); size=70
[outer 1] patch 1: added 6 node(s); size=44
[outer 1] patch 2: added 8 node(s); size=22
[outer 1] patch 3: added 8 node(s); size=31
[outer 1] patch 4: added 6 node(s); size=45
== End outer pass 1: any_added=True, any_merged=False, num_patches=5, total_nodes_in_patches=212
[outer 2] patch 0: added 0 node(s); size=70
[outer 2] patch 1: added 0 node(s); size=44
[outer 2] patch 2: added 0 node(s); size=22
[outer 2] patch 3: added 0 node(s); size=31
[outer 2] patch 4: added 0 node(s); size=45
== End outer pass 2: any_added=False, any_merged=False, num_patches=5, total_nodes_in_patches=212
Growth stabilized (patch-outer).
Patch 0: stopped — insufficient connections (need ≥2). Best above-threshold node 161 HKS=-0.0

In [133]:
import numpy as np
import networkx as nx


def find_regions_from_mask(G, mask, min_region_size=5):
    """
    Connected components induced by nodes where mask is True.
    Keep only components with at least min_region_size nodes.

    Returns
    -------
    regions : list of sets
        Each set contains node indices of a connected component (size-filtered).
    labels : np.ndarray (int, shape (N,))
        For all nodes: -1 if not in any returned region, else the region id (0..K-1).
        (Assumes nodes are labeled 0..N-1 as elsewhere in your pipeline.)
    """
    import numpy as np
    import networkx as nx

    if G.number_of_nodes() != mask.shape[0]:
        raise ValueError("G and mask size mismatch.")

    nodes = [n for n in G.nodes if mask[n]]
    sub = G.subgraph(nodes)
    comps = [set(c) for c in nx.connected_components(sub)]
    regions = [c for c in comps if len(c) >= min_region_size]

    labels = -1 * np.ones(G.number_of_nodes(), dtype=int)
    for rid, reg in enumerate(regions):
        for u in reg:
            labels[u] = rid

    return regions, labels



def get_frontier_candidates_sorted_generic(G, patch_set, labels, HKS_t0, sign):
    # Sort by descending score after sign flip: S(u) = sign * HKS_t0[u]
    cand = list(frontier_nodes(G, patch_set, labels))
    cand.sort(key=lambda u: sign * HKS_t0[u], reverse=True)
    return cand

def diagnose_patch_stall_generic(G, patch_set, labels, HKS_t0, thr_t0, sign):
    """
    Generic version:
      Condition is sign*HKS >= sign*thr.
      Reports whether there are no frontier nodes, none meeting threshold,
      or none with enough neighbors.
    """
    cand = get_frontier_candidates_sorted_generic(G, patch_set, labels, HKS_t0, sign)
    S = lambda u: sign * HKS_t0[u]
    S_thr = sign * thr_t0

    report = {
        "reason": None,
        "details": "",
        "next_node": None,
        "next_node_hks": None,
        "next_node_neighbors_in_patch": None,
        "num_frontier": len(cand),
        "num_frontier_meet_thr": 0,
        "best_frontier_hks": None,
        "sign": int(sign),
        "thr_t0": float(thr_t0),
    }

    if len(cand) == 0:
        report["reason"] = "no_frontier"
        report["details"] = "No unlabeled neighbors adjacent to the patch."
        return report

    best = cand[0]
    report["best_frontier_hks"] = float(HKS_t0[best])

    meet = [u for u in cand if S(u) >= S_thr]
    report["num_frontier_meet_thr"] = len(meet)

    if len(meet) == 0:
        report["reason"] = "no_candidate_meets_threshold"
        # The “next best” is just the best frontier by score S(u)
        report["next_node"] = int(best)
        report["next_node_hks"] = float(HKS_t0[best])
        report["next_node_neighbors_in_patch"] = int(count_neighbors_in_set(G, best, patch_set))
        ineq = "≥" if sign > 0 else "≤"
        report["details"] = (
            f"No frontier nodes meet threshold (HKS {ineq} {thr_t0}). "
            f"Best frontier node {report['next_node']} has HKS={report['next_node_hks']:.4f}."
        )
        return report

    # Some meet threshold but none satisfy neighbor constraint
    top_ok = meet[0]
    nn = count_neighbors_in_set(G, top_ok, patch_set)
    report["reason"] = "insufficient_connections"
    report["next_node"] = int(top_ok)
    report["next_node_hks"] = float(HKS_t0[top_ok])
    report["next_node_neighbors_in_patch"] = int(nn)
    report["details"] = (
        f"Frontier nodes meeting threshold exist but none have enough neighbors "
        f"(need ≥2; best has {nn})."
    )
    return report

def pretty_print_stall_report_generic(patch_id, report):
    ineq = "≥" if report.get("sign", 1) > 0 else "≤"
    reason = report["reason"]
    if reason == "no_frontier":
        msg = f"Patch {patch_id}: stopped — no frontier neighbors."
    elif reason == "no_candidate_meets_threshold":
        msg = (f"Patch {patch_id}: stopped — no frontier nodes meet threshold "
               f"(HKS {ineq} {report['thr_t0']}). Best frontier node "
               f"{report['next_node']} HKS={report['next_node_hks']:.4f} "
               f"(neighbors_in_patch={report['next_node_neighbors_in_patch']}).")
    elif reason == "insufficient_connections":
        msg = (f"Patch {patch_id}: stopped — insufficient connections (need ≥2). "
               f"Best above-threshold node {report['next_node']} "
               f"HKS={report['next_node_hks']:.4f}, "
               f"neighbors_in_patch={report['next_node_neighbors_in_patch']}. "
               f"(candidates_meet_thr={report['num_frontier_meet_thr']})")
    else:
        msg = f"Patch {patch_id}: stopped — reason unknown."
    print(msg)
    return msg


import numpy as np
import networkx as nx

# ---------- (unchanged) helpers you already have ----------
# init_patch_labels, count_neighbors_in_set, merge_touching_patches
# diagnose_patch_stall_generic, pretty_print_stall_report_generic
# If you don't have them in your scope, keep your originals.

def init_patch_labels(n_nodes, regions):
    labels = -1 * np.ones(n_nodes, dtype=int)
    patch_sets = []
    for pid, reg in enumerate(regions):
        s = set(reg)
        patch_sets.append(s)
        for u in s:
            labels[u] = pid
    return labels, patch_sets

def count_neighbors_in_set(G, node, node_set):
    c = 0
    for v in G.neighbors(node):
        if v in node_set:
            c += 1
    return c

def merge_touching_patches(G, labels, patch_sets):
    if len(patch_sets) <= 1:
        return False, labels, patch_sets

    parent = {pid: pid for pid in range(len(patch_sets))}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for u, v in G.edges():
        lu, lv = labels[u], labels[v]
        if lu >= 0 and lv >= 0 and lu != lv:
            union(lu, lv)

    reps = {pid: find(pid) for pid in range(len(patch_sets))}
    if all(pid == reps[pid] for pid in reps):
        return False, labels, patch_sets

    groups = {}
    for old_id, rep in reps.items():
        groups.setdefault(rep, []).append(old_id)

    new_sets = []
    for _new_pid, rep in enumerate(groups.keys()):
        merged_set = set()
        for old_id in groups[rep]:
            merged_set |= patch_sets[old_id]
        new_sets.append(merged_set)

    new_labels = -1 * np.ones_like(labels)
    for new_pid, s in enumerate(new_sets):
        for u in s:
            new_labels[u] = new_pid

    return True, new_labels, new_sets

# ---------- NEW: frontier that respects "blocked" nodes ----------
def frontier_nodes_blocked(G, patch_nodes, labels, blocked_labels=None):
    """
    Frontier = unlabeled neighbors of patch_nodes.
    If blocked_labels is provided, exclude nodes with blocked_labels[v] >= 0.
    """
    cand = set()
    for u in patch_nodes:
        for v in G.neighbors(u):
            if labels[v] == -1:
                if blocked_labels is not None and blocked_labels[v] >= 0:
                    continue  # claimed by the other family
                cand.add(v)
    return cand

# ---------- Generic single-step grow (handles crypts/necks via 'sign') ----------
def grow_one_step_for_patch_generic_with_pid(
    G,
    patch_sets,
    pid,
    labels,
    HKS_t0,
    thr_t0,
    sign=+1,                    # +1: HKS >= thr ; -1: HKS <= thr
    min_neighbors_in_patch=2,
    blocked_labels=None         # labels array from the OTHER family, or None
):
    patch = patch_sets[pid]
    cand = list(frontier_nodes_blocked(G, patch, labels, blocked_labels))
    if not cand:
        return None

    S = lambda u: sign * HKS_t0[u]
    S_thr = sign * thr_t0
    cand.sort(key=lambda u: S(u), reverse=True)  # best first after sign flip

    for u in cand:
        if S(u) >= S_thr and count_neighbors_in_set(G, u, patch) >= min_neighbors_in_patch:
            patch.add(u)
            labels[u] = pid
            return u
    return None

# ---------- Generic patch-outer growth (now with 'blocked_labels') ----------
def grow_patches_until_stable_patch_outer_generic(
    G,
    regions,
    HKS,
    thr_t0,
    sign=+1,                        # +1 crypts, -1 necks
    min_neighbors_in_patch=2,
    max_outer_passes=100,
    max_inner_steps_per_patch=None,
    verbose=True,
    do_diagnostics=True,
    blocked_labels=None             # labels array from the OTHER family, or None
):
    """
    Grow patches while forbidding additions of nodes already claimed by the other family.
    """
    n = G.number_of_nodes()
    HKS_t0 = HKS[:, 0]
    labels, patch_sets = init_patch_labels(n, regions)

    for outer in range(max_outer_passes):
        any_added_in_pass = False
        any_merged_in_pass = False

        pid = 0
        while pid < len(patch_sets):
            patch = patch_sets[pid]
            if not patch:
                pid += 1
                continue

            added_this_patch = 0
            while True:
                added = grow_one_step_for_patch_generic_with_pid(
                    G=G,
                    patch_sets=patch_sets,
                    pid=pid,
                    labels=labels,
                    HKS_t0=HKS_t0,
                    thr_t0=thr_t0,
                    sign=sign,
                    min_neighbors_in_patch=min_neighbors_in_patch,
                    blocked_labels=blocked_labels
                )
                if added is None:
                    break
                added_this_patch += 1
                any_added_in_pass = True
                if max_inner_steps_per_patch is not None and added_this_patch >= max_inner_steps_per_patch:
                    break

            merged, labels, patch_sets = merge_touching_patches(G, labels, patch_sets)
            if merged:
                any_merged_in_pass = True
                pid = 0
                if verbose:
                    print(f"[outer {outer+1}] merged; restart. num_patches={len(patch_sets)}")
                continue

            if verbose:
                print(f"[outer {outer+1}] patch {pid}: +{added_this_patch} nodes; size={len(patch_sets[pid])}")
            pid += 1

        if verbose:
            total_size = sum(len(s) for s in patch_sets)
            print(f"== End outer pass {outer+1}: any_added={any_added_in_pass}, "
                  f"any_merged={any_merged_in_pass}, patches={len(patch_sets)}, "
                  f"nodes_in_patches={total_size}")

        if not any_added_in_pass and not any_merged_in_pass:
            if verbose:
                print("Growth stabilized (generic patch-outer).")
            break

    # Optional diagnostics (reusing your generic versions if you have them)
    stall_reports = None
    if do_diagnostics and 'diagnose_patch_stall_generic' in globals():
        stall_reports = []
        for pid, patch in enumerate(patch_sets):
            rpt = diagnose_patch_stall_generic(G, patch, labels, HKS_t0, thr_t0, sign)
            if 'pretty_print_stall_report_generic' in globals():
                pretty_print_stall_report_generic(pid, rpt)
            stall_reports.append(rpt)

    return (labels, patch_sets, stall_reports) if do_diagnostics else (labels, patch_sets)


import numpy as np
import networkx as nx

def find_neck_loops(
    G,
    neck_labels,               # np.ndarray (N,): -1 not neck, >=0 neck region id
    crypt_labels,              # np.ndarray (N,): -1 not crypt, >=0 crypt region id
    min_loop_len=5,
    max_loop_len=20,           # loops must have length < max_loop_len
    allow_gap=False            # if True, connect boundary neck nodes via ONE crypt node
):
    """
    Identify one 'neck-loop' per boundary region with the constraint:
      - Loop nodes are NECK nodes that are also BOUNDARY-TO-CRYPT
        (each loop node is neck AND has at least one neighbor that is crypt).
      - For each connected boundary component, choose ONE simple cycle with
        min_loop_len <= length < max_loop_len.
      - If allow_gap=True, connect two boundary neck nodes (u,v) if there exists
        a crypt node w with edges (u,w) and (w,v). (Gap passes ONLY through crypt.)

    Returns
    -------
    loops : list of np.ndarray
        One loop per boundary region (if any), each as a sequence of node ids.
    boundary_regions : list of sets
        Connected components of the boundary neck graph used to extract loops.
    """
    N = G.number_of_nodes()
    if neck_labels.shape[0] != N or crypt_labels.shape[0] != N:
        raise ValueError("Label arrays must match number of graph nodes.")

    is_neck  = (neck_labels  >= 0)
    is_crypt = (crypt_labels >= 0)

    # 1) Boundary neck nodes: neck nodes that touch at least one crypt neighbor
    boundary = set()
    for u in G.nodes():
        if not is_neck[u]:
            continue
        for v in G.neighbors(u):
            if is_crypt[v]:          # boundary is strictly to crypt
                boundary.add(u)
                break

    if not boundary:
        return [], []

    # 2) Build boundary graph H over boundary neck nodes
    H = nx.Graph()
    H.add_nodes_from(boundary)

    # Direct boundary edges (neck–neck edges that both are boundary-to-crypt)
    for u, v in G.edges():
        if u in boundary and v in boundary:
            H.add_edge(u, v, gap=0)

    # Optional: one-gap via a single CRYPT node only
    if allow_gap:
        for u in list(boundary):
            for w in G.neighbors(u):
                if not is_crypt[w]:
                    continue  # the intermediate must be crypt
                for v in G.neighbors(w):
                    if v != u and v in boundary:
                        if H.has_edge(u, v):
                            if H[u][v].get("gap", 0) > 0:
                                H[u][v]["gap"] = 0
                        else:
                            H.add_edge(u, v, gap=1)

    # 3) Boundary regions = connected components in H
    boundary_regions = [set(c) for c in nx.connected_components(H)]

    # 4) Extract at most one loop per boundary region (prefer the longest under cap)
    loops = []
    for reg in boundary_regions:
        if len(reg) < min_loop_len:
            continue
        subH = H.subgraph(reg).copy()
        cycles = nx.cycle_basis(subH)  # simple cycles as lists of node ids
        cand = [c for c in cycles if len(c) >= min_loop_len and len(c) < max_loop_len]
        if not cand:
            continue
        best = max(cand, key=lambda c: len(c))  # prefer longest (avoids tiny triangles)
        loops.append(np.asarray(best, dtype=int))

    return loops, boundary_regions


# ---------- helpers: boundary info (crypt-only) ----------


def labels_to_regions(G, labels):
    """
    From an integer labels array (>=0 = in family, -1 = not), compute connected components
    and return (regions, remapped_labels). Regions are sets of node ids. remapped_labels
    is a fresh 0..K-1 per connected component (others -1).
    """
    N = G.number_of_nodes()
    nodes = [n for n in G.nodes() if labels[n] >= 0]
    sub = G.subgraph(nodes)
    comps = [set(c) for c in nx.connected_components(sub)]
    new_labels = -1 * np.ones(N, dtype=int)
    for rid, comp in enumerate(comps):
        for u in comp:
            new_labels[u] = rid
    return comps, new_labels


def _neck_boundary_nodes_crypt_only(G, neck_labels, crypt_labels):
    is_neck  = (neck_labels  >= 0)
    is_crypt = (crypt_labels >= 0)
    boundary = set()
    for u in G.nodes():
        if not is_neck[u]:
            continue
        for v in G.neighbors(u):
            if is_crypt[v]:
                boundary.add(u)
                break
    return boundary

def _sum_loop_lengths(G, neck_labels, crypt_labels, min_loop_len=5, max_loop_len=20, allow_gap=False):
    # Uses your neck-loop finder that restricts to crypt-only boundary
    loops, boundary_regions = find_neck_loops(
        G=G,
        neck_labels=neck_labels,
        crypt_labels=crypt_labels,
        min_loop_len=min_loop_len,
        max_loop_len=max_loop_len,
        allow_gap=allow_gap
    )
    total_len = sum(len(c) for c in loops)
    had_loop = [True for _ in loops]  # presence only; we use count to check "unbroken"
    return total_len, len(loops), loops, boundary_regions

# ---------- helper: check if assigning node u to crypt would merge two crypt regions ----------

def _would_merge_distinct_crypts(G, u, crypt_labels):
    nbr_labels = set(crypt_labels[v] for v in G.neighbors(u) if crypt_labels[v] >= 0)
    return len(nbr_labels) >= 2  # would bridge two distinct crypt regions

def _modal_adjacent_crypt_label(G, u, crypt_labels):
    counts = {}
    for v in G.neighbors(u):
        lab = crypt_labels[v]
        if lab >= 0:
            counts[lab] = counts.get(lab, 0) + 1
    if not counts:
        return None
    # return the label with highest count (ties arbitrary)
    return max(counts.items(), key=lambda kv: kv[1])[0]

# ---------- one-step shrink (removes at most one neck node) ----------

def shrink_neck_once(
    G,
    HKS,                 # (N,T) or at least (N,1); we use column 0
    neck_labels,         # np.ndarray (N,) int: -1 not neck, >=0 neck region id
    crypt_labels,        # np.ndarray (N,) int: -1 not crypt, >=0 crypt region id
    thr_core_keep,       # nodes with HKS_t0 <= thr_core_keep are protected (cannot remove)
    min_loop_len=5,
    max_loop_len=20,
    allow_gap=False
):
    """
    Try to remove exactly one neck boundary node (highest HKS first) subject to rules 1-4.
    On success, returns (True, new_neck_labels, new_crypt_labels, info_dict).
    If no valid removal exists, returns (False, neck_labels, crypt_labels, info_dict).
    """
    N = G.number_of_nodes()
    HKS_t0 = HKS[:, 0]
    # Pre state
    pre_total_len, pre_num_loops, pre_loops, pre_bregions = _sum_loop_lengths(
        G, neck_labels, crypt_labels, min_loop_len, max_loop_len, allow_gap
    )

    # Collect boundary neck nodes (crypt-only boundary) and sort by descending HKS
    boundary = _neck_boundary_nodes_crypt_only(G, neck_labels, crypt_labels)
    cand = sorted(boundary, key=lambda u: HKS_t0[u], reverse=True)

    for u in cand:
        # Rule 1: protect core
        if HKS_t0[u] <= thr_core_keep:
            continue

        # Decide replacement by neighbor majority: crypt vs unlabeled
        nbrs = list(G.neighbors(u))
        nbr_crypt = sum(1 for v in nbrs if crypt_labels[v] >= 0)
        nbr_unlab = sum(1 for v in nbrs if crypt_labels[v] < 0 and neck_labels[v] < 0)
        # majority pick; tie -> unlabeled
        prefer_crypt = (nbr_crypt > nbr_unlab)

        # If prefer crypt, ensure Rule 4 (no crypt merge); else we may skip
        if prefer_crypt and _would_merge_distinct_crypts(G, u, crypt_labels):
            # would connect distinct crypt regions -> cannot remove u to crypt
            continue

        # Tentative new labels
        new_neck = neck_labels.copy()
        new_crypt = crypt_labels.copy()

        # remove from neck
        old_neck_id = new_neck[u]
        new_neck[u] = -1

        # assign to crypt if chosen and safe, else leave unlabeled
        if prefer_crypt:
            lab = _modal_adjacent_crypt_label(G, u, new_crypt)
            if lab is None:
                # No adjacent crypt after all -> fallback to unlabeled
                pass
            else:
                new_crypt[u] = lab

        # Rule 2 & 3: boundary loop total length must not increase; loops must persist
        post_total_len, post_num_loops, post_loops, post_bregions = _sum_loop_lengths(
            G, new_neck, new_crypt, min_loop_len, max_loop_len, allow_gap
        )

        if post_num_loops == 0 and pre_num_loops > 0:
            # broke all loops
            continue
        if post_total_len > pre_total_len:
            # loop length grew
            continue

        # Optional: ensure we didn't increase the number of boundary components drastically;
        # but the main checks above already enforce "not broken" and non-increase.

        # Passed all checks -> commit this removal
        info = {
            "removed_node": int(u),
            "removed_node_hks": float(HKS_t0[u]),
            "prefer_crypt": bool(prefer_crypt),
            "pre_total_loop_len": int(pre_total_len),
            "post_total_loop_len": int(post_total_len),
            "pre_num_loops": int(pre_num_loops),
            "post_num_loops": int(post_num_loops),
            "old_neck_id": int(old_neck_id) if old_neck_id >= 0 else -1
        }
        return True, new_neck, new_crypt, info

    # No candidate could be removed
    return False, neck_labels, crypt_labels, {
        "removed_node": None,
        "reason": "no_valid_candidate",
        "pre_total_loop_len": int(pre_total_len),
        "pre_num_loops": int(pre_num_loops)
    }

# ---------- multi-step shrink until stable (or max_steps) ----------

def shrink_neck_until_stable(
    G,
    HKS,
    neck_labels,
    crypt_labels,
    thr_core_keep,
    min_loop_len=5,
    max_loop_len=20,
    allow_gap=False,
    max_steps=10,
    verbose=True
):
    """
    Repeatedly call shrink_neck_once() to shave the neck a node at a time.
    Stops when no valid removal exists or max_steps reached.

    Returns
    -------
    cur_neck_labels : np.ndarray (N,)
    cur_crypt_labels : np.ndarray (N,)
    neck_patches : list[set[int]]
        Connected components of current neck (labels >= 0).
    crypt_patches : list[set[int]]
        Connected components of current crypt (labels >= 0).
    info_list : list[dict]
        Step-by-step logs (last item explains why it stopped if no removal).
    """
    steps = 0
    info_list = []
    cur_neck = neck_labels.copy()
    cur_crypt = crypt_labels.copy()

    while steps < max_steps:
        changed, new_neck, new_crypt, info = shrink_neck_once(
            G, HKS, cur_neck, cur_crypt, thr_core_keep,
            min_loop_len=min_loop_len, max_loop_len=max_loop_len, allow_gap=allow_gap
        )
        info_list.append(info)
        if not changed:
            if verbose:
                print("Neck shrinking stabilized (no valid removal).")
            break
        steps += 1
        cur_neck, cur_crypt = new_neck, new_crypt
        if verbose:
            print(f"[shrink step {steps}] removed node {info['removed_node']} "
                  f"(HKS={info['removed_node_hks']:.4f}), "
                  f"prefer_crypt={info['prefer_crypt']}, "
                  f"loops {info['pre_total_loop_len']}→{info['post_total_loop_len']}.")

    # Build patches from final labels (connected components), and also remap labels accordingly
    neck_patches, cur_neck_remapped = labels_to_regions(G, cur_neck)
    crypt_patches, cur_crypt_remapped = labels_to_regions(G, cur_crypt)

    # Return remapped labels (consistent with patches) for downstream use
    return cur_neck_remapped, cur_crypt_remapped, neck_patches, crypt_patches, info_list

def labels_to_regions(G, labels):
    """
    From an integer labels array (>=0=in family, -1=not), compute connected components
    and return (regions, remapped_labels). Regions are sets of node ids. remapped_labels
    is a fresh 0..K-1 per connected component (others -1).
    """
    N = G.number_of_nodes()
    nodes = [n for n in G.nodes() if labels[n] >= 0]
    sub = G.subgraph(nodes)
    comps = [set(c) for c in nx.connected_components(sub)]
    new_labels = -1 * np.ones(N, dtype=int)
    for rid, comp in enumerate(comps):
        for u in comp:
            new_labels[u] = rid
    return comps, new_labels

def promote_unlabeled_components_to_crypt(
    G,
    crypt_labels,
    neck_labels=None,
    min_unlabeled_size=11  # >10 nodes
):
    """
    Promote all unlabeled connected components (w.r.t. the graph) of size >= min_unlabeled_size
    to crypt patches.

    Unlabeled = nodes that are NOT crypt and (if neck_labels provided) NOT neck.

    Parameters
    ----------
    G : nx.Graph
    crypt_labels : np.ndarray (N,) int
        -1 for non-crypt, >=0 for crypt patch id.
    neck_labels : np.ndarray (N,) int or None
        If provided, nodes with neck_labels>=0 are excluded from promotion.
    min_unlabeled_size : int
        Absolute minimum size of unlabeled component to promote (default 11 => >10).

    Returns
    -------
    new_crypt_labels : np.ndarray (N,) int
        Updated crypt labels, remapped to 0..K-1 after promotion.
    crypt_patches : list[set]
        Connected components of crypt after promotion.
    """
    N = G.number_of_nodes()
    if crypt_labels.shape[0] != N:
        raise ValueError("crypt_labels length must match number of nodes.")
    if neck_labels is not None and neck_labels.shape[0] != N:
        raise ValueError("neck_labels length must match number of nodes.")

    # Determine unlabeled mask (not crypt, and if provided, not neck)
    is_crypt = (crypt_labels >= 0)
    if neck_labels is not None:
        is_neck = (neck_labels >= 0)
        unlabeled = (~is_crypt) & (~is_neck)
    else:
        unlabeled = (~is_crypt)

    # Find unlabeled connected components
    unlabeled_nodes = [n for n in G.nodes() if unlabeled[n]]
    sub = G.subgraph(unlabeled_nodes)
    comps = [set(c) for c in nx.connected_components(sub)]

    # Start with a copy of crypt labels; assign a fresh id to each promoted comp
    new_labels = crypt_labels.copy()
    next_id = (int(np.max(new_labels)) + 1) if np.any(new_labels >= 0) else 0

    promoted = 0
    for comp in comps:
        if len(comp) >= min_unlabeled_size:
            for u in comp:
                new_labels[u] = next_id
            next_id += 1
            promoted += 1

    # Remap to connected components (ensures contiguous ids and merges if touching)
    crypt_patches, new_labels = labels_to_regions(G, new_labels)

    # Optional: you can print or log how many were promoted
    # print(f"Promoted {promoted} unlabeled component(s) (>= {min_unlabeled_size}) to crypt.")

    return new_labels, crypt_patches



def detect_seeds_crypt_and_neck(
    coords,
    G,
    HKS,
    thr_allpos,          # crypt seed threshold (all time-scales >=)
    thr_allneg,          # neck  seed threshold (all time-scales <=)
    thr_grow_t0,
    min_region_size=5,
    plot_func=None,
    title="Crypt & Neck seeds (debug)"
):
    """
    Streamlined: return only final crypts and necks as lists of node-index sets.
    Labels are used internally for blocking and post-processing.

    Returns
    -------
    dict with:
        "crypts" : list[set[int]]   # final crypt patches (connected components)
        "necks"  : list[set[int]]   # final neck  patches (connected components)
        "figure" : plotly Figure or None
    """
    # --- Seeds (all-scales thresholds) ---
    high_mask_all = make_high_mask(HKS, thr_allpos=thr_allpos)  # crypt seeds
    # tolerate either function name for the low-all-times mask
    try:
        low_mask_all = make_low_mask(HKS, thr_allneg=thr_allneg)      # neck seeds (if you defined this name)
    except Exception:
        low_mask_all = make_low_mask_all(HKS, thr_allneg=thr_allneg)  # neck seeds (alternate name)

    crypt_seed_regions, crypt_seed_labels = find_regions_from_mask(G, high_mask_all, min_region_size=min_region_size)
    neck_seed_regions,  neck_seed_labels  = find_regions_from_mask(G, low_mask_all,  min_region_size=min_region_size)

    # --- Grow necks first (so they can block crypt growth) ---
    neck_labels, neck_patches, _ = grow_patches_until_stable_patch_outer_generic(
        G=G, regions=neck_seed_regions, HKS=HKS,
        thr_t0=thr_grow_t0,
        sign=-1,                           # necks: HKS <= thr
        min_neighbors_in_patch=2,
        blocked_labels=crypt_seed_labels,  # forbid annexing crypt seeds
        verbose=True
    )

    # --- Grow crypts (block with grown necks) ---
    crypt_labels, crypt_patches, _ = grow_patches_until_stable_patch_outer_generic(
        G=G, regions=crypt_seed_regions, HKS=HKS,
        thr_t0=thr_grow_t0,                # use the same growth thr; tune if desired
        sign=+1,                           # crypts: HKS >= thr
        min_neighbors_in_patch=2,
        blocked_labels=neck_labels,        # forbid annexing neck cells
        verbose=True
    )

    # --- Promote large unlabeled components to crypts (e.g., villus) ---
    crypt_labels, crypt_patches = promote_unlabeled_components_to_crypt(
        G=G,
        crypt_labels=crypt_labels,
        neck_labels=neck_labels,
        min_unlabeled_size=11              # >10 nodes
    )

    # --- Shrink necks while preserving topology (may also adjust crypt labels) ---
    neck_labels, crypt_labels, neck_patches, crypt_patches, _ = shrink_neck_until_stable(
        G=G,
        HKS=HKS,
        neck_labels=neck_labels,
        crypt_labels=crypt_labels,
        thr_core_keep=-1,    # protect very negative HKS nodes from removal; tune to your scale
        min_loop_len=5,
        max_loop_len=100,
        allow_gap=False,
        max_steps=1000,
        verbose=True
    )

    # --- Plot (optional): overlays expect lists[set[int]] ---
    fig = None
    if plot_func is not None:
        z_small = HKS[:, 0]
        fig = plot_func(coords=coords, G=G, z_small=z_small, loops=[], title=title)
        add_crypt_overlays(fig, coords, crypt_patches)
        add_neck_overlays(fig, coords, neck_patches)

    # Final, minimal outputs: lists of sets (no labels)
    return {
        "crypts": crypt_patches,
        "necks": neck_patches,
        "figure": fig
    }



# HKS_zscored = (cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))/np.std(cell_weighted_hks, axis=0)

curvature_proxy = HKS_times*(cell_weighted_hks-np.mean(cell_weighted_hks, axis=0))
curvature_proxy = curvature_proxy/np.std(curvature_proxy, axis=0)


result = detect_seeds_crypt_and_neck(
    coords=centroids,
    G=cell_graph,
    HKS=curvature_proxy[:,:3],
    thr_allpos=0.6,   # crypt seed threshold (all scales >= 0.15)
    thr_allneg=-1.00,  # neck seed threshold (all scales <= -0.05)
    thr_grow_t0=-0.1,
    min_region_size=8,
    plot_func=plot_organoid_loops_3d,
    title="Seeds: crypts (warm) & necks (cool)"
)

print(len(result["crypts"]))


fig = result["figure"]
if fig is not None:
    fig.show()

[outer 1] patch 0: +98 nodes; size=122
[outer 1] patch 1: +76 nodes; size=150
== End outer pass 1: any_added=True, any_merged=False, patches=2, nodes_in_patches=272
[outer 2] patch 0: +0 nodes; size=122
[outer 2] patch 1: +0 nodes; size=150
== End outer pass 2: any_added=False, any_merged=False, patches=2, nodes_in_patches=272
Growth stabilized (generic patch-outer).
Patch 0: stopped — insufficient connections (need ≥2). Best above-threshold node 746 HKS=-0.3829, neighbors_in_patch=1. (candidates_meet_thr=3)
Patch 1: stopped — insufficient connections (need ≥2). Best above-threshold node 283 HKS=-0.9093, neighbors_in_patch=1. (candidates_meet_thr=6)
[outer 1] patch 0: +16 nodes; size=69
[outer 1] patch 1: +14 nodes; size=48
[outer 1] patch 2: +11 nodes; size=31
[outer 1] patch 3: +9 nodes; size=44
== End outer pass 1: any_added=True, any_merged=False, patches=4, nodes_in_patches=192
[outer 2] patch 0: +0 nodes; size=69
[outer 2] patch 1: +0 nodes; size=48
[outer 2] patch 2: +0 nodes; s